# Hands-on LLM: Transformers, Attention & Fine-tuning

Run this notebook **top to bottom** in Google Colab. It starts with vectors and self-attention using real NumPy calculations, then trains a tiny PyTorch model, and finally offers an optional FLAN-T5 probe.

The toy examples are illustrative rather than pretrained linguistic representations. The optional model section downloads `google/flan-t5-base` and therefore takes more time and memory.

## 0 — Running in Google Colab

No local Python or Jupyter installation is required. Select **Runtime → Run all**, or run one cell at a time with `Shift + Return`. Colab provides NumPy, Matplotlib, and PyTorch in its hosted runtime.

A numbered prompt means the cell finished. A spinning indicator means it is still running. The optional FLAN-T5 cell installs its own pinned packages only when you set `RUN_FLAN_T5 = True`.

## 1 — Your very first code cell

In [ ]:
print('Hello, Jupyter!')
print('Two plus two =', 2 + 2)

## 2 — What environment is running this cell?

In [ ]:
import sys
print('Python:', sys.version.split()[0])
print('Interpreter:', sys.executable)

## 3 — Vectors and cosine similarity

A vector is just an ordered list of numbers. Cosine similarity compares direction. These example vectors are **not** model-learned embeddings.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([10.0, 20.0])
c = np.array([-2.0, 1.0])

def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))

print('cos(a,b) =', round(cosine(a,b), 4))  # 1: same direction
print('cos(a,c) =', round(cosine(a,c), 4))  # 0: perpendicular


## 4 — Draw vectors in two dimensions

This plot is faithful to these 2D vectors. Real FLAN-T5 embeddings have **768 dimensions** and cannot be plotted literally in 2D without projection.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 5))
for name, vec in [('a', a), ('b', b), ('c', c)]:
    ax.arrow(0, 0, vec[0], vec[1], head_width=.35, length_includes_head=True)
    ax.text(vec[0], vec[1], name, fontsize=12)
ax.axhline(0, linewidth=.5); ax.axvline(0, linewidth=.5)
ax.set_xlim(-4, 12); ax.set_ylim(-4, 23)
ax.set_aspect('equal'); ax.grid(alpha=.2)
plt.show()

## 5 — Three toy tokens, two dimensions

Each row represents one token position, each column a feature. We use identity-like projections so the numbers stay interpretable.

In [ ]:
X = np.array([
    [1.0, 0.0],    # teacher
    [0.0, 1.0],    # gave
    [1.0, 1.0],    # book
])
Wq = np.eye(2)
Wk = np.eye(2)
Wv = np.array([[1.0, .5], [.5, 1.0]])
Q = X @ Wq
K = X @ Wk
V = X @ Wv
print('X shape:', X.shape)
print('Q:\n', Q)
print('K:\n', K)
print('V:\n', V)

## 6 — Inspect the complete attention calculation

For this toy example, `d_k=2`. A production FLAN-T5 Base encoder head uses `d_k=64`.

In [ ]:
scores = Q @ K.T / np.sqrt(Q.shape[1])
stable = scores - scores.max(axis=-1, keepdims=True)
exp_scores = np.exp(stable)
weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
context = weights @ V
print('Scaled scores:\n', np.round(scores, 3))
print('Attention weights (rows sum to 1):\n', np.round(weights, 3))
print('Row sums:', weights.sum(axis=1))
print('Context vectors:\n', np.round(context, 3))

## 7 — Visualize attention as a heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(3), ['teacher', 'gave', 'book'])
ax.set_yticks(range(3), ['teacher', 'gave', 'book'])
ax.set_xlabel('Key token'); ax.set_ylabel('Query token')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', color='white')
fig.colorbar(im)
plt.show()

## 8 — Build a tiny trainable model (PyTorch)

This is a minimal embedding + output projection model for a one-token next-token target, **not yet a Transformer**. The purpose is to see weights change during training.

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
vocab = ['teacher', 'gave', 'student', 'book', '<eos>']
embedding = nn.Embedding(len(vocab), 8)
output_layer = nn.Linear(8, len(vocab))
optimizer = torch.optim.AdamW(list(embedding.parameters()) + list(output_layer.parameters()), lr=.05)
input_id = torch.tensor([2])  # student
target_id = torch.tensor([3])  # book

before = embedding.weight.detach().clone()
for step in range(5):
    optimizer.zero_grad()
    logits = output_layer(embedding(input_id))
    loss = nn.functional.cross_entropy(logits, target_id)
    loss.backward()
    optimizer.step()
    print(f'step={step+1} loss={loss.item():.4f}')

after = embedding.weight.detach()
print('Did stored weights change?', not torch.equal(before, after))

## 9 — Optional FLAN-T5 Base inspection

This section is disabled by default. It downloads roughly 1 GB of model files and uses more RAM than the toy examples. Set `RUN_FLAN_T5 = True` in the next cell to install the pinned dependencies and inspect the pretrained model. The cell performs inference setup only and does not change the pretrained weights.

In [ ]:
RUN_FLAN_T5 = False

if RUN_FLAN_T5:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers==4.38.2', 'sentencepiece', 'protobuf'
    ])
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    model_name = 'google/flan-t5-base'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    original_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, torch_dtype=torch.float32
    )
    print('Total model parameters:', sum(p.numel() for p in original_model.parameters()))
    print('book token IDs:', tokenizer('book', add_special_tokens=False)['input_ids'])
else:
    print('Optional FLAN-T5 inspection skipped. Set RUN_FLAN_T5 = True to run it.')

## 10 — Where to go next

Experiment with different token vectors, change the attention projection matrices, and compare how the heatmap changes. Then study a full dialogue-summarization workflow to see pretrained encoder-decoder attention, fine-tuning, LoRA, and evaluation in a realistic task.